## R Setup

In [ ]:
# Load libraries
library(tableone)
library(Matching)
library(ipw)
library(survey)
library(MatchIt)
library(rbounds)
library(cobalt)

## Pre-processing

In [ ]:
data(lalonde)
head(lalonde)

In [ ]:
# See all column names
names(lalonde)

In [ ]:
# See structure
str(lalonde)

In [ ]:
lalonde$black  <- ifelse(lalonde$race == "black", 1, 0)
lalonde$hispan <- ifelse(lalonde$race == "hispan", 1, 0)

## Exploratory Data Analysis

In [ ]:
# Distribution plots: age, earnings, propensity scores
par(mfrow = c(2, 3))
hist(lalonde$age,  main = "Age",        col = "skyblue")
hist(lalonde$educ, main = "Education",  col = "salmon")
hist(lalonde$re74, main = "Earnings 74", col = "lightgreen")
hist(lalonde$re75, main = "Earnings 75", col = "lightyellow")
hist(lalonde$re78, main = "Earnings 78 (Outcome)", col = "plum")

In [ ]:
%%R
# Outcome by treatment group (before any adjustment)
boxplot(re78 ~ treat, data = lalonde,
        names = c("Control", "Treated"),
        main = "Re78 by Treatment (Unadjusted)",
        ylab = "Real Earnings 1978", col = c("tomato", "steelblue"))

## Fit propensity score model

In [ ]:
# # Define confounding variables
# xvars <- c("age", "educ", "race", "married",
#            "nodegree", "re74", "re75")

# # Define which variables are categorical
# catvars <- c("race", "married", "nodegree")

xvars <- c("age", "educ", "black", "hispan", "married",
           "nodegree", "re74", "re75")

catvars <- c("black", "hispan", "married", "nodegree")

# Create pre-matching Table 1 with SMD
table1 <- CreateTableOne(vars = xvars,
                         factorVars = catvars,
                         strata = "treat",
                         data = lalonde,
                         test = FALSE)

# Print with SMD
print(table1, smd = TRUE)

In [ ]:
# Calculate raw mean difference in re78 (treated - untreated)
mean_treated <- mean(lalonde$re78[lalonde$treat == 1])
mean_control <- mean(lalonde$re78[lalonde$treat == 0])

raw_diff <- mean_treated - mean_control
round(raw_diff, 2)

In [ ]:
# Logistic regression model with all 8 confounders
ps_model <- glm(treat ~ age + educ + race + married +
                nodegree + re74 + re75,
                data = lalonde,
                family = binomial())

# Get propensity scores for each subject
lalonde$ps <- predict(ps_model, type = "response")

# Min and max of propensity scores
summary(lalonde$ps)
# min(lalonde$ps)
# max(lalonde$ps)

## Propensity Score Diagnostics

In [ ]:
# Overlap / common support plot
par(mfrow = c(1, 2))
hist(lalonde$ps[lalonde$treat == 1], breaks = 20, col = rgb(0,0,1,0.5),
     main = "PS Distribution", xlab = "Propensity Score", xlim = c(0,1))
hist(lalonde$ps[lalonde$treat == 0], breaks = 20, col = rgb(1,0,0,0.5), add = TRUE)
legend("topright", c("Treated", "Control"), fill = c(rgb(0,0,1,0.5), rgb(1,0,0,0.5)))

# Love plot (SMD before vs after matching) — via cobalt
love.plot(match_out, data = lalonde, treat = lalonde$treat,
          covs = lalonde[, xvars], threshold = 0.1, stars = "raw")

## Propensity score matching

In [ ]:
set.seed(931139)

# Match using propensity score
# - pair matching (M=1)
# - without replacement (replace=FALSE)
# - no caliper
# - match on ps itself (not logit)
match_out <- Match(Tr = lalonde$treat,
                   M = 1,
                   X = lalonde$ps,
                   replace = FALSE,
                   caliper = NULL)

# Extract matched data
matched_data <- lalonde[unlist(match_out[c("index.treated",
                                           "index.control")]), ]

# Check balance on matched data
table1_matched <- CreateTableOne(vars = xvars,
                                 factorVars = catvars,
                                 strata = "treat",
                                 data = matched_data,
                                 test = FALSE)

# Print with SMD — look for married row
print(table1_matched, smd = TRUE)

## Largest SMD in matched data (no caliper)

In [ ]:
# Just inspect the SMD column visually — look for largest value
print(table1_matched, smd = TRUE)

## Re-match WITH caliper = 0.1

In [ ]:
set.seed(931139)

match_caliper <- Match(Tr = lalonde$treat,
                       M = 1,
                       X = lalonde$ps,
                       replace = FALSE,
                       caliper = 0.1)

# How many matched pairs?
match_caliper$wnobs

## Mean difference in re78 for caliper-matched data

In [ ]:
# Extract treated and control outcomes using matched indices
re78_treated  <- lalonde$re78[match_caliper$index.treated]
re78_control  <- lalonde$re78[match_caliper$index.control]

mean(re78_treated) - mean(re78_control)

## Paired t-test on caliper-matched data

In [ ]:
# Paired t-test (respects the matched pair structure)
t_result <- t.test(re78_treated, re78_control, paired = TRUE)
t_result

# Just the 95% CI
t_result$conf.int

IPTW creates a pseudo-population where confounders are balanced between treated and control units. Each subject is weighted by the inverse of the probability of receiving the treatment they actually received.

## IPTW weights

In [ ]:
lalonde$iptw <- ifelse(lalonde$treat == 1,
                       1/lalonde$ps,
                       1/(1 - lalonde$ps))

print(min(lalonde$iptw))
print(max(lalonde$iptw))

## Weighted balance table

In [ ]:
svy_design <- svydesign(ids = ~1,
                        weights = ~iptw,
                        data = lalonde)

weighted_table <- svyCreateTableOne(vars = xvars,
                                    strata = "treat",
                                    data = svy_design,
                                    factorVars = catvars,
                                    test = FALSE)
print(weighted_table, smd = TRUE)


## IPTW causal effect via svyglm

In [ ]:
msm <- svyglm(re78 ~ treat,
              design = svydesign(~1, weights = ~iptw, data = lalonde))

# svyglm already has robust SEs built in — just use confint directly
coef(msm)["treat"]
confint(msm, "treat")

## Truncated weights at 1st/99th percentile

In [ ]:
# Truncate weights at 1st/99th percentile
p01 <- quantile(lalonde$iptw, 0.01)
p99 <- quantile(lalonde$iptw, 0.99)
lalonde$iptw_trunc <- pmin(pmax(lalonde$iptw, p01), p99)

cat("Original range: ", min(lalonde$iptw), "-", max(lalonde$iptw), "\n")
cat("Truncated range:", min(lalonde$iptw_trunc), "-", max(lalonde$iptw_trunc), "\n")

### Results Summary


In [ ]:
# Calculate PSM (no caliper) estimate and CI
mean_treated_nocal <- mean(matched_data$re78[matched_data$treat == 1])
mean_control_nocal <- mean(matched_data$re78[matched_data$treat == 0])
mean_diff_psm_nocal <- mean_treated_nocal - mean_control_nocal

# Perform paired t-test for PSM (no caliper)
ttest_nocal <- t.test(matched_data$re78[matched_data$treat == 1],
                        matched_data$re78[matched_data$treat == 0],
                        paired = TRUE)
ci_lower_nocal <- as.numeric(ttest_nocal$conf.int[1])
ci_upper_nocal <- as.numeric(ttest_nocal$conf.int[2])

# Assign PSM (caliper=0.1) estimate to a variable
mean_diff_psm_caliper <- mean(re78_treated) - mean(re78_control)

# Assign PSM (caliper=0.1) CI to variables
ci_lower_caliper <- as.numeric(t_result$conf.int[1])
ci_upper_caliper <- as.numeric(t_result$conf.int[2])

# Assign IPTW estimate and CI to variables
estimate_iptw <- as.numeric(coef(msm)["treat"])
ci_iptw <- confint(msm, "treat")
ci_lower_iptw <- as.numeric(ci_iptw[1])
ci_upper_iptw <- as.numeric(ci_iptw[2])

# Calculate IPTW (truncated) estimate and CI
msm_trunc <- svyglm(re78 ~ treat,
                     design = svydesign(~1, weights = ~iptw_trunc, data = lalonde))
estimate_iptw_trunc <- as.numeric(coef(msm_trunc)["treat"])
ci_iptw_trunc <- confint(msm_trunc, "treat")
ci_lower_iptw_trunc <- as.numeric(ci_iptw_trunc[1])
ci_upper_iptw_trunc <- as.numeric(ci_iptw_trunc[2])

# Create the results data frame
results <- data.frame(
  Method              = c("Unadjusted", "PSM (no caliper)", "PSM (caliper=0.1)", "IPTW", "IPTW (truncated)"),
  Estimate            = c(raw_diff, mean_diff_psm_nocal, mean_diff_psm_caliper, estimate_iptw, estimate_iptw_trunc),
  CI_Lower            = c(NA, ci_lower_nocal, ci_lower_caliper, ci_lower_iptw, ci_lower_iptw_trunc),
  CI_Upper            = c(NA, ci_upper_nocal, ci_upper_caliper, ci_upper_iptw, ci_upper_iptw_trunc)
)
print(results)

### Sensitivity Analysis

In [ ]:
# Rosenbaum bounds: how strong would unmeasured confounding need to be
# to explain away the treatment effect?
psens(re78_treated, re78_control, Gamma = 2, GammaInc = 0.1)